# PIE Dataset Setup — Week 1 (v2: direct video to HDF5)

Revised to avoid extracting 300k+ loose JPEGs to Drive, decodes frames straight from video, crops to the pedestrian bbox, packs into a few HDF5 files. No loose-file I/O bottleneck on the Drive mount.

Run top to bottom. Start the video download (Cell 3) first, it's the slow part.

## 1. Mount Drive, folders, clone PIE repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

DRIVE_ROOT = '/content/drive/MyDrive/PIE_thesis'
PIE_DATA_ROOT = f'{DRIVE_ROOT}/PIE_dataset'
SPLITS_DIR = f'{DRIVE_ROOT}/splits'
CROPS_DIR = f'{DRIVE_ROOT}/crops'   # packed HDF5 files go here, not loose images

for d in [DRIVE_ROOT, PIE_DATA_ROOT, SPLITS_DIR, CROPS_DIR]:
    os.makedirs(d, exist_ok=True)

%cd /content
!git clone https://github.com/aras62/PIE.git PIE_repo
sys.path.append('/content/PIE_repo')


Mounted at /content/drive
/content
Cloning into 'PIE_repo'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 181 (delta 35), reused 75 (delta 18), pack-reused 85 (from 1)
Receiving objects: 100% (181/181), 144.63 MiB | 16.10 MiB/s, done.
Resolving deltas: 100% (76/76), done.


## 2. Download video clips (74GB, start now, runs in background)

In [6]:
!cat /content/PIE_repo/download_clips.sh


cat: /content/PIE_repo/download_clips.sh: No such file or directory


In [2]:
# Correct path: script lives in annotations/, downloads to current dir via wget
os.makedirs(PIE_DATA_ROOT, exist_ok=True)
%cd {PIE_DATA_ROOT}
!bash /content/PIE_repo/annotations/download_clips.sh

/content/drive/MyDrive/PIE_thesis/PIE_dataset
--2026-10-08 08:23:32--  https://data.nvision2.eecs.yorku.ca/PIE_dataset/PIE_clips/
Resolving data.nvision2.eecs.yorku.ca (data.nvision2.eecs.yorku.ca)... 130.63.94.247
Connecting to data.nvision2.eecs.yorku.ca (data.nvision2.eecs.yorku.ca)|130.63.94.247|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘PIE_clips/index.html.tmp’

PIE_clips/index.htm     [ <=>                ]     845  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-10-08 08:23:33 (86.1 MB/s) - ‘PIE_clips/index.html.tmp’ saved [845]

Loading robots.txt; please ignore errors.
--2026-10-08 08:23:33--  https://data.nvision2.eecs.yorku.ca/robots.txt
Reusing existing connection to data.nvision2.eecs.yorku.ca:443.
HTTP request sent, awaiting response... 404 Not Found
2026-10-08 08:23:33 ERROR 404: Not Found.

Removing PIE_clips/index.html.tmp since it should be rejected.

--2026-10-08

In [8]:
!cat /content/PIE_repo/annotations/download_clips.sh

# Download PIE clips

wget -N --recursive --no-parent -nH --cut-dirs=1 -R "index.html*" https://data.nvision2.eecs.yorku.ca/PIE_dataset/PIE_clips/ .


In [7]:
!ls -la /content/PIE_repo
!find /content/PIE_repo -iname '*download*'

total 52
drwxr-xr-x 10 root root 4096 Oct  8 08:10 .
drwxr-xr-x  1 root root 4096 Oct  8 08:10 ..
drwxr-xr-x  2 root root 4096 Oct  8 08:10 annotations
drwxr-xr-x  2 root root 4096 Oct  8 08:10 camera_params
drwxr-xr-x  2 root root 4096 Oct  8 08:10 demos
drwxr-xr-x  8 root root 4096 Oct  8 08:10 .git
-rw-r--r--  1 root root 1087 Oct  8 08:10 LICENSE
drwxr-xr-x  2 root root 4096 Oct  8 08:10 model_outputs
-rw-r--r--  1 root root 2650 Oct  8 08:10 README.md
-rw-r--r--  1 root root   90 Oct  8 08:10 requirements.txt
drwxr-xr-x  2 root root 4096 Oct  8 08:10 scenarioEval
drwxr-xr-x  2 root root 4096 Oct  8 08:10 utilities
drwxr-xr-x  2 root root 4096 Oct  8 08:10 visualization
/content/PIE_repo/annotations/download_clips.sh


## 3. Verify PIE API and load default-split sequences

In [23]:
import os, sys
os.path.exists('/content/PIE_repo') or get_ipython().system('git clone https://github.com/aras62/PIE.git /content/PIE_repo')
('/content/PIE_repo/utilities' in sys.path) or sys.path.append('/content/PIE_repo/utilities')
[os.remove(f'{PIE_DATA_ROOT}/annotations/{f}') for f in os.listdir(f'{PIE_DATA_ROOT}/annotations') if not os.path.isdir(f'{PIE_DATA_ROOT}/annotations/{f}')]
from pie_data import PIE
imdb = PIE(data_path=PIE_DATA_ROOT)
data_opts = {'seq_type': 'crossing', 'data_split_type': 'default'}
train_seqs = imdb.generate_data_trajectory_sequence('train', **data_opts)
val_seqs = imdb.generate_data_trajectory_sequence('val', **data_opts)
test_seqs = imdb.generate_data_trajectory_sequence('test', **data_opts)
print('Available keys:', list(train_seqs.keys()))
print('train/val/test sequences:', len(train_seqs['pid']), len(val_seqs['pid']), len(test_seqs['pid']))

---------------------------------------------------------
Generating trajectory sequence data
fstride: 1
sample_type: all
height_rng: [0, inf]
squarify_ratio: 0
data_split_type: default
seq_type: crossing
min_track_size: 15
random_params: {'ratios': None, 'val_data': True, 'regen_data': False}
kfold_params: {'num_folds': 5, 'fold': 1}
---------------------------------------------------------
Generating database for pie
pie annotations loaded from /content/drive/MyDrive/PIE_thesis/PIE_dataset/data_cache/pie_database.pkl
---------------------------------------------------------
Generating crossing data
Excluding tracks with less than 15
Subset: train
Number of pedestrians: 880 
Total number of samples: 876 
---------------------------------------------------------
Generating trajectory sequence data
fstride: 1
sample_type: all
height_rng: [0, inf]
squarify_ratio: 0
data_split_type: default
seq_type: crossing
min_track_size: 15
random_params: {'ratios': None, 'val_data': True, 'regen_data

In [16]:
p = f'{PIE_DATA_ROOT}/annotations/set01/video_0001_annt.xml'
print(os.path.exists(p), repr(p))
print(sorted(os.listdir(f'{PIE_DATA_ROOT}/annotations/set01')))

True '/content/drive/MyDrive/PIE_thesis/PIE_dataset/annotations/set01/video_0001_annt.xml'
['set01_annotated_frames.csv', 'video_0001_annt.xml', 'video_0002_annt.xml', 'video_0003_annt.xml', 'video_0004_annt.xml']


In [18]:
get_ipython().system('mv ' + PIE_DATA_ROOT + '/annotations/annotations_attributes ' + PIE_DATA_ROOT + '/annotations_attributes')
get_ipython().system('mv ' + PIE_DATA_ROOT + '/annotations/annotations_vehicle ' + PIE_DATA_ROOT + '/annotations_vehicle')
get_ipython().system('rm -rf ' + PIE_DATA_ROOT + '/data_cache')
print(sorted(os.listdir(PIE_DATA_ROOT)))
print(sorted(os.listdir(f'{PIE_DATA_ROOT}/annotations')))

['PIE_clips', 'annotations', 'annotations_attributes', 'annotations_vehicle', 'camera_params']
['set01', 'set02', 'set03', 'set04', 'set05', 'set06']


In [7]:
os.path.exists(f'{PIE_DATA_ROOT}/annotations') or get_ipython().system('cp -r /content/PIE_repo/annotations ' + PIE_DATA_ROOT + '/annotations')
os.path.exists(f'{PIE_DATA_ROOT}/camera_params') or get_ipython().system('cp -r /content/PIE_repo/camera_params ' + PIE_DATA_ROOT + '/camera_params')
print(sorted(os.listdir(PIE_DATA_ROOT)))

['PIE_clips', 'annotations', 'camera_params', 'data_cache']


In [10]:
print('annotations dir:', sorted(os.listdir(f'{PIE_DATA_ROOT}/annotations')))
import pickle
with open(f'{PIE_DATA_ROOT}/data_cache/pie_database.pkl', 'rb') as f:
    db = pickle.load(f)
    print('database keys:', sorted(db.keys()))

annotations dir: []
database keys: []


In [11]:
import subprocess
print(subprocess.run(['find', '/content/PIE_repo', '-iname', '*annt*'], capture_output=True, text=True).stdout)
print(subprocess.run(['find', '/content/PIE_repo', '-iname', '*.xml'], capture_output=True, text=True).stdout[:500])
print(subprocess.run(['find', '/content/PIE_repo', '-iname', '*.zip'], capture_output=True, text=True).stdout)



/content/PIE_repo/annotations/annotations_attributes.zip
/content/PIE_repo/annotations/annotations_vehicle.zip
/content/PIE_repo/annotations/annotations.zip



In [12]:
get_ipython().system('rm -rf ' + PIE_DATA_ROOT + '/annotations ' + PIE_DATA_ROOT + '/data_cache')
os.makedirs(f'{PIE_DATA_ROOT}/annotations', exist_ok=True)
get_ipython().system('cp /content/PIE_repo/annotations/*.zip ' + PIE_DATA_ROOT + '/annotations/')
get_ipython().system('cd ' + PIE_DATA_ROOT + '/annotations && unzip -q -o annotations.zip && unzip -q -o annotations_attributes.zip && unzip -q -o annotations_vehicle.zip')
get_ipython().system('rm ' + PIE_DATA_ROOT + '/annotations/*.zip')
print(sorted(os.listdir(f'{PIE_DATA_ROOT}/annotations')))

['annotations', 'annotations_attributes', 'annotations_vehicle']


In [13]:
print(sorted(os.listdir(f'{PIE_DATA_ROOT}/annotations/annotations')))

['set01', 'set02', 'set03', 'set04', 'set05', 'set06']


In [14]:
get_ipython().system('mv ' + PIE_DATA_ROOT + '/annotations/annotations/* ' + PIE_DATA_ROOT + '/annotations/')
get_ipython().system('rmdir ' + PIE_DATA_ROOT + '/annotations/annotations')
print(sorted(os.listdir(f'{PIE_DATA_ROOT}/annotations')))
print(sorted(os.listdir(f'{PIE_DATA_ROOT}/annotations/set01'))[:5])

['annotations_attributes', 'annotations_vehicle', 'set01', 'set02', 'set03', 'set04', 'set05', 'set06']
['set01_annotated_frames.csv', 'video_0001_annt.xml', 'video_0002_annt.xml', 'video_0003_annt.xml', 'video_0004_annt.xml']


## 4. Confirm the crossing label key and class distribution

Same honesty check as before, confirm `CROSS_KEY` and the irrelevant-label value against what actually prints, don't assume.

In [20]:
import numpy as np

CROSS_KEY = 'activities'  # confirm/adjust against the keys printed above
IRRELEVANT_VALUE = -1     # confirm against the distribution below

labels = train_seqs[CROSS_KEY]
flat_labels = [item for seq in labels for item in (seq if isinstance(seq, list) else [seq])]
unique, counts = np.unique(flat_labels, return_counts=True)
print('Raw distribution:', dict(zip(unique.tolist(), counts.tolist())))


Raw distribution: {0: 199846, 1: 78601}


## 5. Confirm video file naming before decoding

Check what's actually in `PIE_clips` before writing a path resolver, don't assume a naming pattern.

In [21]:
clips_root = f'{PIE_DATA_ROOT}/PIE_clips'
for root, dirs, files in os.walk(clips_root):
    depth = root[len(clips_root):].count(os.sep)
    if depth <= 1:
        print(root, '->', sorted(dirs)[:5], sorted(files)[:5])
    if depth > 1:
        break


/content/drive/MyDrive/PIE_thesis/PIE_dataset/PIE_clips -> ['set01', 'set02', 'set03', 'set04', 'set05'] []
/content/drive/MyDrive/PIE_thesis/PIE_dataset/PIE_clips/set01 -> [] ['video_0001.mp4', 'video_0002.mp4', 'video_0003.mp4', 'video_0004.mp4']
/content/drive/MyDrive/PIE_thesis/PIE_dataset/PIE_clips/set02 -> [] ['video_0001.mp4', 'video_0002.mp4', 'video_0003.mp4']
/content/drive/MyDrive/PIE_thesis/PIE_dataset/PIE_clips/set03 -> [] ['video_0001.mp4', 'video_0002.mp4', 'video_0003.mp4', 'video_0004.mp4', 'video_0005.mp4']
/content/drive/MyDrive/PIE_thesis/PIE_dataset/PIE_clips/set04 -> [] ['video_0001.mp4', 'video_0002.mp4', 'video_0003.mp4', 'video_0004.mp4', 'video_0005.mp4']
/content/drive/MyDrive/PIE_thesis/PIE_dataset/PIE_clips/set05 -> [] ['video_0001.mp4', 'video_0002.mp4']
/content/drive/MyDrive/PIE_thesis/PIE_dataset/PIE_clips/set06 -> [] ['video_0001.mp4', 'video_0002.mp4', 'video_0003.mp4', 'video_0004.mp4', 'video_0005.mp4']


## 6. Extract bbox crops directly from video into HDF5 (no loose files)

Uses whatever frame-reference key PIE exposes (commonly `image`, containing a path that encodes set/video/frame) to seek the exact frame with OpenCV, crop to bbox, resize, and append to one HDF5 file per split. Only the final frame of each sequence is pulled (Model A's input per the thesis plan); Model B's full 2.0s window can reuse this same function with a frame-range later this week.

**If Cell 5's output shows a different folder/file naming than assumed in `_resolve_video_path()`, fix that function before running this.**

In [24]:
import cv2
import h5py
import re

IMG_SIZE = 224

def _resolve_video_path(image_ref):
    """image_ref is expected to look like .../images/set01/video_0001/00123.png
    Adjust this parsing if Cell 5 showed a different pattern."""
    parts = image_ref.replace('\\', '/').split('/')
    set_id = next((p for p in parts if re.match(r'set\d+', p)), None)
    video_id = next((p for p in parts if re.match(r'video_\d+', p)), None)
    frame_num = int(re.sub(r'\D', '', parts[-1]))
    video_path = f'{clips_root}/{set_id}/{video_id}.mp4'
    return video_path, frame_num

def extract_split_to_hdf5(seqs, cross_key, irrelevant_value, out_path):
    n_written = 0
    video_cache = {}

    with h5py.File(out_path, 'w') as hf:
        images_ds = hf.create_dataset('images', shape=(0, IMG_SIZE, IMG_SIZE, 3),
                                       maxshape=(None, IMG_SIZE, IMG_SIZE, 3), dtype='uint8')
        labels_ds = hf.create_dataset('labels', shape=(0,), maxshape=(None,), dtype='int64')

        for i in range(len(seqs['pid'])):
            labels = seqs[cross_key][i]
            label = labels
            while isinstance(label, list): label = label[-1]
            if label == irrelevant_value:
                continue
            if 'image' not in seqs or 'bbox' not in seqs:
                raise KeyError('Expected "image" and "bbox" keys not found, check Cell 3 output and adjust key names here.')

            image_ref = seqs['image'][i][-1]
            bbox = seqs['bbox'][i][-1]
            video_path, frame_num = _resolve_video_path(image_ref)

            if video_path not in video_cache:
                video_cache[video_path] = cv2.VideoCapture(video_path)
            cap = video_cache[video_path]
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ok, frame = cap.read()
            if not ok:
                continue

            x1, y1, x2, y2 = [int(v) for v in bbox]
            crop = frame[max(y1,0):y2, max(x1,0):x2]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
            crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

            images_ds.resize(n_written + 1, axis=0)
            labels_ds.resize(n_written + 1, axis=0)
            images_ds[n_written] = crop
            labels_ds[n_written] = int(label)
            n_written += 1

    for cap in video_cache.values():
        cap.release()

    print(f'Wrote {n_written} crops to {out_path}')
    return n_written

for split_name, seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    out_path = f'{CROPS_DIR}/{split_name}_crops.h5'
    extract_split_to_hdf5(seqs, CROSS_KEY, IRRELEVANT_VALUE, out_path)


Wrote 876 crops to /content/drive/MyDrive/PIE_thesis/crops/train_crops.h5
Wrote 243 crops to /content/drive/MyDrive/PIE_thesis/crops/val_crops.h5
Wrote 717 crops to /content/drive/MyDrive/PIE_thesis/crops/test_crops.h5


## 7. Document the split by pedestrian ID

In [27]:
import pandas as pd

def _unwrap(p):
    return _unwrap(p[0]) if isinstance(p, list) else p
split_records = []
for split_name, seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    pids = sorted(set(_unwrap(p) for p in seqs['pid']))
    for pid in pids: split_records.append({'split': split_name, 'pedestrian_id': pid})
split_df = pd.DataFrame(split_records)
split_df.to_csv(f'{SPLITS_DIR}/pie_default_split_pedestrian_ids.csv', index=False)
split_df['split'].value_counts()

,count
split,
train,876
test,717
val,243


## 8. Commit to GitHub

```bash
cd pedestrian-crossing-prediction
cp <SPLITS_DIR>/pie_default_split_pedestrian_ids.csv results/
git add notebooks/week1_data_setup.ipynb results/pie_default_split_pedestrian_ids.csv
git commit -m "Week 1: PIE setup, direct video-to-HDF5 bbox crop extraction, split documentation"
git push
```

The HDF5 files in `CROPS_DIR` stay on Drive, never in git, Model A's notebook reads directly from them next.